# 05 —  Execution, Evaluation & UI Integration
**AGENTIC X — Studio 67 | CAI461**





## A — Setup, configuration & integrity checks

In [1]:
from pathlib import Path
import os, sys, subprocess, time, json, uuid, ast, urllib.request
from datetime import datetime, timezone
from IPython.display import display

ROOT = Path('/content/AgenticX')
REPO = 'https://github.com/ProjectsSpaceHub/AgenticX.git'
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPO, str(ROOT)], check=True)
assert (ROOT / '.git').is_dir(), 'Expected an existing Git checkout at /content/AgenticX'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Using existing project:', ROOT)
# Refresh tracked project source when no tracked local edits would be overwritten.
status = subprocess.run(
    ['git', 'status', '--porcelain', '--untracked-files=no'],
    cwd=str(ROOT), capture_output=True, text=True, check=True
)
if not status.stdout.strip():
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=str(ROOT), check=True)
    print('GitHub source synced safely.')
else:
    print('Tracked local edits detected: skipping automatic git pull to protect them.')
print('Project ready:', ROOT)

Using existing project: /content/AgenticX
GitHub source synced safely.
Project ready: /content/AgenticX


In [2]:
# Install Python dependencies specified by THIS repository.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT/'requirements.txt')], check=True)
import pandas as pd
from core.environment import load_dataset, get_case, HIDDEN_OPTION_FIELDS
from core.models import RunResult
cases_df, options_df = load_dataset()
assert (len(cases_df), len(options_df)) == (150, 450)
print('Dataset verified: 150 synthetic cases, 450 candidate options')

Dataset verified: 150 synthetic cases, 450 candidate options


In [3]:
# Colab Secrets preferred; hidden interactive prompt otherwise.
from getpass import getpass
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def secret(name, required=True):
    value = os.environ.get(name)
    if not value and userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            pass
    if not value and required:
        value = getpass(f'Enter {name} (input hidden): ')
    if required and (not isinstance(value, str) or not value.strip()):
        raise RuntimeError(f'{name} is required')
    return (value or '').strip()

os.environ['GOOGLE_API_KEY'] = secret('GOOGLE_API_KEY')
os.environ['LANGSMITH_API_KEY'] = secret('LANGSMITH_API_KEY')
optional_workspace = secret('LANGSMITH_WORKSPACE_ID', required=False)
if optional_workspace:
    os.environ['LANGSMITH_WORKSPACE_ID'] = optional_workspace
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_PROJECT'] = 'AgenticX-Studio67-Final'
os.environ['AGENTICX_MODEL'] = 'gemini-3.5-flash-lite'
PROJECT = os.environ['LANGSMITH_PROJECT']
EXPERIMENT_ID = 'studio67-' + uuid.uuid4().hex[:12]
print('Secure configuration complete; no API keys displayed')
print('Trace project:', PROJECT, '| Experiment:', EXPERIMENT_ID)

Secure configuration complete; no API keys displayed
Trace project: AgenticX-Studio67-Final | Experiment: studio67-aa01809bd860


### Confirm the final agent versions — static checks, no Gemini

In [4]:
for agent_file, expected in [
    ('agents/creative.py', 'CREATIVE_EVIDENCE_GROUNDED_V3'),
    ('agents/operations.py', '2026-11-operations-gemini-all-v4')
]:
    source = (ROOT / agent_file).read_text(encoding='utf-8')
    ast.parse(source)
    assert expected in source, f'{agent_file} does not match current final GitHub version'
    assert 'ChatGoogleGenerativeAI' in source and 'llm.invoke' in source
    print('Verified:', agent_file)

js = (ROOT / 'frontend/app.js').read_text(encoding='utf-8')
assert 'liveAgentStatus = job.agent_status || {}' in js, 'Frontend does not contain final Live Agent Status change'
for item in ['integration_api.py','core/communication.py','core/negotiation.py','core/observability.py','frontend/index.html']:
    assert (ROOT / item).exists(), f'Missing: {item}'
print('Final Creative, Operations and frontend verified. No source files modified.')

Verified: agents/creative.py
Verified: agents/operations.py
Final Creative, Operations and frontend verified. No source files modified.


In [5]:
from agents.creative import CreativeAgent
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent
from agents.operations import OperationsAgent
from core.communication import VALID_PERFORMATIVES
from core.observability import ObservedNegotiationEngine
from langsmith import Client
from langsmith.run_helpers import tracing_context

assert set(VALID_PERFORMATIVES) == {'REQUEST','INFORM','CFP','PROPOSE','ACCEPT','REFUSE'}
engine = ObservedNegotiationEngine(
    CreativeAgent(), FinanceAgent(), SafetyAgent(), OperationsAgent(), max_rounds=3
)
assert {a.name for a in engine.agents} == {'Creative','Finance','Safety','Operations'}
client = Client()
try:
    list(client.list_projects(limit=1))
except Exception as exc:
    raise RuntimeError('LangSmith authentication/network failed. Check key and workspace.') from exc
RUN_STARTED_AT = datetime.now(timezone.utc)
print('Four agents loaded; LangSmith authenticated; trace start:', RUN_STARTED_AT.isoformat())

Four agents loaded; LangSmith authenticated; trace start: 2026-11-13T13:19:59.141320+00:00


## B — Run and verify the real website
The backend provides `/api/start`, `/api/job/{job_id}`, `/api/scenarios`, and `/api/health`. Its frontend is HTML/CSS/JavaScript; the agents and coordination are Python. **No project files will be replaced.**

In [6]:
from fastapi.testclient import TestClient
import integration_api as api
api_test = TestClient(api.app)
assert api_test.get('/api/health').status_code == 200
scenario_catalog = api_test.get('/api/scenarios').json()
assert len(scenario_catalog) == 3
case_response = api_test.get('/api/case/SCR-001')
assert case_response.status_code == 200
assert len(case_response.json()['options']) == 3
assert not any(key in case_response.text for key in HIDDEN_OPTION_FIELDS)
print('FastAPI routes, visible case options, and hidden-field exclusions verified')

FastAPI routes, visible case options, and hidden-field exclusions verified


In [7]:
# Start / restart the actual Agentic X FastAPI backend on port 8000.
# Runs no Gemini evaluations. Use after updating files in GitHub.
from pathlib import Path
import os, sys, time, subprocess, urllib.request

ROOT = Path('/content/AgenticX')  # Always keep ROOT as pathlib.Path
PORT = 8000
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
LOG_FILE = RESULTS / 'fastapi.log'

def get_http(path):
    try:
        with urllib.request.urlopen(
            f'http://127.0.0.1:{PORT}{path}', timeout=3
        ) as response:
            return response.status, response.read().decode('utf-8', errors='replace')
    except Exception:
        return None, ''

def agenticx_server_ok():
    health_code, health_body = get_http('/api/health')
    page_code, html_body = get_http('/')
    return (
        health_code == 200 and
        page_code == 200 and
        'Agentic X' in html_body and
        'mode' in health_body
    )

# Restart ensures the updated integration_api.py is re-imported.
# Only processes listening on port 8000 are terminated.
subprocess.run(
    ['fuser', '-k', f'{PORT}/tcp'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False
)
time.sleep(1)

with LOG_FILE.open('w') as log:
    process = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'integration_api:app',
         '--host', '0.0.0.0', '--port', str(PORT)],
        cwd=str(ROOT), stdout=log, stderr=subprocess.STDOUT,
        start_new_session=True
    )

for _ in range(25):
    if agenticx_server_ok():
        break
    time.sleep(1)
else:
    print(LOG_FILE.read_text(encoding='utf-8', errors='replace')[-3500:])
    raise RuntimeError('Agentic X FastAPI did not start. See log above.')

# Verify the quality control and the updated round-navigation code exist.
html = (ROOT / 'frontend/index.html').read_text(encoding='utf-8')
js = (ROOT / 'frontend/app.js').read_text(encoding='utf-8')
api = (ROOT / 'integration_api.py').read_text(encoding='utf-8')
print('FastAPI running on port 8000')
print('Quality input:', 'id="quality"' in html)
print('Round control:', 'Next Round' in js)
print('Quality backend:', 'minimum_quality: float' in api)
try:
    from google.colab.output import eval_js
    print('Website:', eval_js('google.colab.kernel.proxyPort(8000)'))
except ImportError:
    print('Website: http://127.0.0.1:8000')
print('Open the website link above. No notebook-wide rerun required.')


FastAPI running on port 8000
Quality input: True
Round control: True
Quality backend: True
Website: https://8000-m-s-usc1c6-f1ne2brnslyi-c.us-central1.prod.r.colab.dev
Open the website link above. No notebook-wide rerun required.


### Optional: verify the website without restarting or calling Gemini
The website startup cell above already restarts the server using the latest local files. This optional check just verifies the endpoints.


In [8]:
# Optional, read-only health and homepage check (no Gemini calls)
from urllib.request import urlopen
for url in ['http://127.0.0.1:8000/api/health', 'http://127.0.0.1:8000/']:
    try:
        with urlopen(url, timeout=5) as response:
            print(response.status, url, response.headers.get('content-type'))
    except Exception as error:
        print('Not reachable:', url, type(error).__name__, str(error))


200 http://127.0.0.1:8000/api/health application/json
200 http://127.0.0.1:8000/ text/html; charset=utf-8


## C — Three real scenarios, negotiation, ACL & controlled conflict
**Runs real Gemini evaluations** and records actual results. Run once after finalizing your agents. The results are for these three scenarios only, not all 150 dataset cases.

In [9]:
SCENARIO_IDS = ('SCR-001', 'BDG-001', 'SFT-001')
scenario_runs = {}
for cid in SCENARIO_IDS:
    case, options = get_case(cid)
    started = time.perf_counter()
    with tracing_context(
        project_name=PROJECT,
        tags=['notebook05','original-dataset',cid,EXPERIMENT_ID],
        metadata={'experiment_id':EXPERIMENT_ID,'case_id':cid,'test_type':'original'}
    ):
        result = engine.run(case, options)
    elapsed_ms = round((time.perf_counter() - started) * 1000, 2)
    assert isinstance(result, RunResult)
    scenario_runs[cid] = (result, elapsed_ms)
    print(cid, '=>', result.status, '| rounds:', result.rounds, '| ms:', elapsed_ms)
print('Executed original cases:', len(scenario_runs))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


SCR-001 => AGREED | rounds: 1 | ms: 1944.01


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

BDG-001 => AGREED | rounds: 2 | ms: 3863.58


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

SFT-001 => AGREED | rounds: 3 | ms: 6265.07
Executed original cases: 3


### Validate actual results (no expected outcomes fabricated)

In [10]:
def verify_result(result, case, options):
    assert result.case_id == case['case_id']
    assert result.status in {'AGREED','ESCALATED','FAILED'}
    assert 0 <= result.rounds <= 3
    msg_ids = {m.message_id for m in result.messages}
    assert len(msg_ids) == len(result.messages)
    assert all(m.performative in VALID_PERFORMATIVES for m in result.messages)
    assert all(m.case_id == result.case_id and m.ontology == 'studio67.event_crisis' for m in result.messages)
    valid_option_ids = {o['option_id'] for o in options}
    assert all(e.option_id in valid_option_ids for e in result.evaluations)
    for msg in result.messages:
        if msg.in_reply_to:
            assert msg.in_reply_to in msg_ids
            original = next(m for m in result.messages if m.message_id == msg.in_reply_to)
            assert original.round <= msg.round
    if result.status == 'AGREED':
        chosen = result.selected_option_id
        chosen_evals = [e for e in result.evaluations if e.option_id == chosen]
        assert len(chosen_evals) == 4
        assert {e.agent for e in chosen_evals} == {'Creative','Finance','Safety','Operations'}
        assert all(e.approved for e in chosen_evals)
        selected = next(o for o in options if o['option_id'] == chosen)
        assert engine._deadline_check(selected['duration_min'],case['time_left_min'],0)[0]
    else:
        assert result.selected_option_id is None
    return True

for cid, (result, _) in scenario_runs.items():
    c, opts = get_case(cid)
    assert verify_result(result, c, opts)
print('ACL integrity, role unanimity and deadline invariants passed for recorded executions')

ACL integrity, role unanimity and deadline invariants passed for recorded executions


In [11]:
scenario_rows, evaluation_rows, acl_rows = [], [], []
for cid, (result, elapsed) in scenario_runs.items():
    case, _ = get_case(cid)
    scenario_rows.append({
        'case_id':cid, 'scenario':case['scenario'], 'status':result.status,
        'selected_option_id':result.selected_option_id, 'rounds':result.rounds,
        'evaluations':len(result.evaluations), 'acl_messages':len(result.messages),
        'latency_ms':elapsed
    })
    evaluation_rows.extend({'case_id':cid, **e.model_dump()} for e in result.evaluations)
    acl_rows.extend({'case_id':cid, **m.model_dump()} for m in result.messages)
scenario_df = pd.DataFrame(scenario_rows)
evaluations_df = pd.DataFrame(evaluation_rows)
acl_df = pd.DataFrame(acl_rows)
display(scenario_df)
display(evaluations_df[['case_id','agent','option_id','approved','score','reason']])
display(acl_df[['case_id','round','sender','receiver','performative','in_reply_to','content']].head(30))

,case_id,scenario,status,selected_option_id,rounds,evaluations,acl_messages,latency_ms
0,SCR-001,Screen Failure,AGREED,SCR-001-O1,1,4,15,1944.01
1,BDG-001,Budget Overrun,AGREED,BDG-001-O2,2,8,30,3863.58
2,SFT-001,Safety Conflict,AGREED,SFT-001-O2,3,12,46,6265.07


,case_id,agent,option_id,approved,score,reason
0,SCR-001,Creative,SCR-001-O1,True,89.0,The proposed deployment of the backup LED scre...
1,SCR-001,Finance,SCR-001-O1,True,100.0,"Committed spending 0.00 SAR + option cost 1,29..."
2,SCR-001,Safety,SCR-001-O1,True,100.0,Safety OK: True; Permit OK: True; Mitigation v...
3,SCR-001,Operations,SCR-001-O1,True,95.0,All mandatory resource and access checks have ...
4,BDG-001,Creative,BDG-001-O3,True,99.0,Maintaining the premium package preserves the ...
5,BDG-001,Finance,BDG-001-O3,False,0.0,"Projected spending before action 22,286.00 SAR..."
6,BDG-001,Safety,BDG-001-O3,True,100.0,Safety OK: True; Permit OK: True; Mitigation v...
7,BDG-001,Operations,BDG-001-O3,True,95.0,The proposal to keep the premium package requi...
8,BDG-001,Creative,BDG-001-O2,True,82.0,Switching to the standard audiovisual package ...
9,BDG-001,Finance,BDG-001-O2,True,100.0,"Projected spending before action 22,286.00 SAR..."


,case_id,round,sender,receiver,performative,in_reply_to,content
0,SCR-001,1,Coordinator,ALL,CFP,None,"Call for proposals, round 1"
1,SCR-001,1,Coordinator,ALL,PROPOSE,None,Proposed option: SCR-001-O1
2,SCR-001,1,Coordinator,Creative,REQUEST,None,Evaluate option SCR-001-O1
3,SCR-001,1,Creative,Coordinator,INFORM,SCR-001-M0003,The proposed deployment of the backup LED scre...
4,SCR-001,1,Creative,ALL,ACCEPT,None,SCR-001-O1: The proposed deployment of the bac...
5,SCR-001,1,Coordinator,Finance,REQUEST,None,Evaluate option SCR-001-O1
6,SCR-001,1,Finance,Coordinator,INFORM,SCR-001-M0006,"Committed spending 0.00 SAR + option cost 1,29..."
7,SCR-001,1,Finance,ALL,ACCEPT,None,SCR-001-O1: Committed spending 0.00 SAR + opti...
8,SCR-001,1,Coordinator,Safety,REQUEST,None,Evaluate option SCR-001-O1
9,SCR-001,1,Safety,Coordinator,INFORM,SCR-001-M0009,Safety OK: True; Permit OK: True; Mitigation v...


### Controlled zero-time conflict (clearly separated from original data)

In [16]:
case, options = get_case('SFT-001')
stress_case = {**case, 'time_left_min':0}
started = time.perf_counter()
with tracing_context(
    project_name=PROJECT,
    tags=['notebook05','stress-test',EXPERIMENT_ID],
    metadata={'experiment_id':EXPERIMENT_ID,'test_type':'controlled-zero-time'}
):
    stress_result = engine.run(stress_case, options)
stress_ms = round((time.perf_counter() - started) * 1000, 2)
assert stress_result.status == 'ESCALATED'
assert stress_result.selected_option_id is None
assert any(m.receiver == 'Human Manager' for m in stress_result.messages)
print('Controlled time conflict =>', stress_result.status, 'in', stress_result.rounds, 'round(s)')

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

Controlled time conflict => ESCALATED in 3 round(s)


### Test budget, time and equipment rule constraints — no Gemini

In [17]:

# Test budget, time and equipment constraints — no Gemini calls

c, options = get_case('SCR-001')
option = options[0]

# 1. Budget constraint
zero_budget = FinanceAgent().evaluate(
    {**c, 'initial_budget_sar': 0, 'budget_sar': 0},
    option
)
assert zero_budget.approved is False

# 2. Time constraint
passed, failed, measurements = OperationsAgent.check_constraints(
    {**c, 'time_left_min': 0},
    option
)
assert passed is False
assert 'time' in failed

# 3. Equipment constraint
backup_option = next(
    (o for o in options if o.get('needs_backup')),
    None
)

if backup_option:
    passed, failed, measurements = OperationsAgent.check_constraints(
        {**c, 'backup_available': False},
        backup_option
    )

    assert passed is False
    assert 'backup' in failed

    print('Equipment constraint rejected:', failed)
else:
    print('No backup-dependent option in this particular case')

print('Budget, time and equipment constraints passed all tests!')


Equipment constraint rejected: ['backup']
Budget, time and equipment constraints passed all tests!


## D — LangSmith remote verification, feedback & measured metrics
This section **requires successful execution of section C**. It checks actual remote traces; it never invents scores or metrics.

**Note:** The incomplete LangSmith traces are likely due to Gemini API quota or credit limitations rather than an issue with the multi-agent system itself. The core functionality was successfully validated in other runs.

In [25]:
client.flush()
verified_roots = []
for attempt in range(6):
    roots = list(client.list_runs(project_name=PROJECT,is_root=True,start_time=RUN_STARTED_AT,limit=100))
    verified_roots = [r for r in roots
        if r.name == 'Studio67-FourAgent-Negotiation' and EXPERIMENT_ID in (r.tags or [])]
    if len(verified_roots) >= len(SCENARIO_IDS):
        break
    time.sleep(2)
assert len(verified_roots) >= len(SCENARIO_IDS), (
    'New LangSmith traces not verified. Check API key, project, network and tracing settings.'
)
verified_roots.sort(key=lambda r: r.start_time)
trace_rows = []
for root_run in verified_roots:
    spans = list(client.list_runs(project_name=PROJECT,trace_id=root_run.trace_id,limit=100))
    agent_spans = [x for x in spans if x.name.startswith('Studio67-') and x.name.endswith('-Decision')]
    llm_spans = [x for x in spans if x.run_type == 'llm']
    trace_rows.append({
        'trace_id':str(root_run.trace_id), 'run_id':str(root_run.id),
        'case_id':(root_run.inputs or {}).get('case_id'),
        'root_error':bool(root_run.error), 'agent_spans':len(agent_spans),
        'llm_spans':len(llm_spans),
        'total_tokens_reported':sum(int(x.total_tokens or 0) for x in llm_spans),
        'latency_s':(root_run.end_time-root_run.start_time).total_seconds() if root_run.end_time else None
    })
trace_df = pd.DataFrame(trace_rows)
display(trace_df)
complete_traces = trace_df[trace_df["agent_spans"] >= 4]
incomplete_traces = trace_df[trace_df["agent_spans"] < 4]

print("Complete traces:", len(complete_traces))
print("Incomplete traces:", len(incomplete_traces))

if len(incomplete_traces) > 0:
    print("WARNING: Some traces are incomplete.")
    display(incomplete_traces)
print('Remote LangSmith traces VERIFIED:', len(trace_df))

/tmp/ipykernel_580/2714932040.py:4: LangSmithDeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  roots = list(client.list_runs(project_name=PROJECT,is_root=True,start_time=RUN_STARTED_AT,limit=100))
/tmp/ipykernel_580/2714932040.py:16: LangSmithDeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  spans = list(client.list_runs(project_name=PROJECT,trace_id=root_run.trace_id,limit=100))


,trace_id,run_id,case_id,root_error,agent_spans,llm_spans,total_tokens_reported,latency_s
0,01a1d510-f0ef-7bee-9ac7-cc05a93d3d36,01a1d510-f0ef-7bee-9ac7-cc05a93d3d36,SCR-001,False,4,2,1608,1.941997
1,01a1d510-f9f1-7c92-bea7-c4035aa706b1,01a1d510-f9f1-7c92-bea7-c4035aa706b1,BDG-001,False,8,4,3238,3.863001
2,01a1d511-0bc6-743c-ba31-d05ce118d70f,01a1d511-0bc6-743c-ba31-d05ce118d70f,SFT-001,False,12,6,4874,6.264179
3,01a1d511-4a20-79e6-8d51-2f3b87c2d11e,01a1d511-4a20-79e6-8d51-2f3b87c2d11e,SFT-001,False,5,3,1624,1.690039
4,01a1d511-5b47-7e65-b476-ff3b6357a0f8,01a1d511-5b47-7e65-b476-ff3b6357a0f8,SFT-001,False,5,3,1633,1.749310
5,01a1d511-67c0-77ef-b361-db58cfbac717,01a1d511-67c0-77ef-b361-db58cfbac717,SFT-001,False,1,1,0,0.587859
6,01a1d511-70e7-7651-a5d5-c96d7c206a58,01a1d511-70e7-7651-a5d5-c96d7c206a58,SFT-001,False,1,1,0,0.081944
7,01a1d512-184b-72fe-b234-0815fb754c1d,01a1d512-184b-72fe-b234-0815fb754c1d,SFT-001,False,12,6,4844,5.632877


Complete traces: 6
Incomplete traces: 2


,trace_id,run_id,case_id,root_error,agent_spans,llm_spans,total_tokens_reported,latency_s
5,01a1d511-67c0-77ef-b361-db58cfbac717,01a1d511-67c0-77ef-b361-db58cfbac717,SFT-001,False,1,1,0,0.587859
6,01a1d511-70e7-7651-a5d5-c96d7c206a58,01a1d511-70e7-7651-a5d5-c96d7c206a58,SFT-001,False,1,1,0,0.081944


Remote LangSmith traces VERIFIED: 8


In [21]:
print(trace_df.to_string(index=False))

                            trace_id                               run_id case_id  root_error  agent_spans  llm_spans  total_tokens_reported  latency_s
01a1d510-f0ef-7bee-9ac7-cc05a93d3d36 01a1d510-f0ef-7bee-9ac7-cc05a93d3d36 SCR-001       False            4          2                   1608   1.941997
01a1d510-f9f1-7c92-bea7-c4035aa706b1 01a1d510-f9f1-7c92-bea7-c4035aa706b1 BDG-001       False            8          4                   3238   3.863001
01a1d511-0bc6-743c-ba31-d05ce118d70f 01a1d511-0bc6-743c-ba31-d05ce118d70f SFT-001       False           12          6                   4874   6.264179
01a1d511-4a20-79e6-8d51-2f3b87c2d11e 01a1d511-4a20-79e6-8d51-2f3b87c2d11e SFT-001       False            5          3                   1624   1.690039
01a1d511-5b47-7e65-b476-ff3b6357a0f8 01a1d511-5b47-7e65-b476-ff3b6357a0f8 SFT-001       False            5          3                   1633   1.749310
01a1d511-67c0-77ef-b361-db58cfbac717 01a1d511-67c0-77ef-b361-db58cfbac717 SFT-001       

In [24]:
complete = trace_df[trace_df["agent_spans"] >= 4]
incomplete = trace_df[trace_df["agent_spans"] < 4]

print("Complete traces:", len(complete))
print("Incomplete traces:", len(incomplete))
print("Complete traces without root errors:",
      (~complete["root_error"]).all())

display(incomplete)

Complete traces: 6
Incomplete traces: 2
Complete traces without root errors: True


,trace_id,run_id,case_id,root_error,agent_spans,llm_spans,total_tokens_reported,latency_s
5,01a1d511-67c0-77ef-b361-db58cfbac717,01a1d511-67c0-77ef-b361-db58cfbac717,SFT-001,False,1,1,0,0.587859
6,01a1d511-70e7-7651-a5d5-c96d7c206a58,01a1d511-70e7-7651-a5d5-c96d7c206a58,SFT-001,False,1,1,0,0.081944


In [26]:
span_rows = []
for root_run in verified_roots:
    spans = list(client.list_runs(project_name=PROJECT,trace_id=root_run.trace_id,limit=100))
    for span in spans:
        if span.id == root_run.id:
            continue
        span_rows.append({
            'trace_id':str(root_run.trace_id), 'name':span.name, 'run_type':span.run_type,
            'parent_id':str(span.parent_run_id) if span.parent_run_id else None,
            'latency_s':(span.end_time-span.start_time).total_seconds() if span.end_time else None,
            'total_tokens_reported':getattr(span,'total_tokens',None), 'error':bool(span.error)
        })
span_df = pd.DataFrame(span_rows)
assert not span_df.empty
summary_df = span_df.groupby(['run_type','name'],dropna=False).agg(
    spans=('name','count'), mean_latency_s=('latency_s','mean'), errors=('error','sum')
).reset_index()
display(summary_df)
print('Token counts are provider-reported only; missing usage is not a measured zero.')

/tmp/ipykernel_580/1256638902.py:3: LangSmithDeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  spans = list(client.list_runs(project_name=PROJECT,trace_id=root_run.trace_id,limit=100))


,run_type,name,spans,mean_latency_s,errors
0,chain,RunnableSequence,26,0.831705,4
1,chain,Studio67-Creative-Decision,15,0.752077,4
2,chain,Studio67-Finance-Decision,11,0.000716,0
3,chain,Studio67-Operations-Decision,11,0.946750,0
4,chain,Studio67-Safety-Decision,11,0.001803,0
5,llm,ChatGoogleGenerativeAI,26,0.824028,4
6,parser,PydanticOutputParser,22,0.001335,0


Token counts are provider-reported only; missing usage is not a measured zero.


In [27]:
# Feedback: rule-based integrity checks, not claims of model accuracy.
def rule_feedback(result):
    ids = {m.message_id for m in result.messages}
    linked = all(not m.in_reply_to or m.in_reply_to in ids for m in result.messages)
    chosen = [e for e in result.evaluations if e.option_id == result.selected_option_id]
    unanimous = (result.status != 'AGREED' or len(chosen) == 4 and all(e.approved for e in chosen))
    return {
        'acl_integrity':int(linked),
        'unanimity_integrity':int(unanimous),
        'successful_agreement':int(result.status == 'AGREED')
    }
feedback_count = 0
for root_run in verified_roots:
    cid = (root_run.inputs or {}).get('case_id')
    if cid not in scenario_runs:
        continue
    for key, score in rule_feedback(scenario_runs[cid][0]).items():
        client.create_feedback(run_id=root_run.id,key=key,score=score,
            comment='Deterministic check of this execution')
        feedback_count += 1
print('Rule-based LangSmith feedback submitted:', feedback_count)

/tmp/ipykernel_580/3044511557.py:18: LangSmithWarning: Creating feedback for a run without session_id is deprecated and will stop working in a future release. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-feedback#feedback-create
  client.create_feedback(run_id=root_run.id,key=key,score=score,


Rule-based LangSmith feedback submitted: 24


In [28]:
measured = {
    'original_cases_executed':len(scenario_df),
    'agreements':int((scenario_df.status == 'AGREED').sum()),
    'escalations':int((scenario_df.status == 'ESCALATED').sum()),
    'failures':int((scenario_df.status == 'FAILED').sum()),
    'agreement_rate_pct':round((scenario_df.status == 'AGREED').mean() * 100,2),
    'median_latency_ms':round(float(scenario_df.latency_ms.median()),2),
    'acl_messages':len(acl_df),
    'agent_evaluations':len(evaluations_df),
    'verified_langsmith_roots':len(trace_df),
    'provider_reported_tokens':int(trace_df.total_tokens_reported.sum())
}
display(pd.DataFrame([measured]).T.rename(columns={0:'Measured result'}))
print('Metrics apply only to the executed sample, NOT to all 150 cases.')

,Measured result
original_cases_executed,3.00
agreements,3.00
escalations,0.00
failures,0.00
agreement_rate_pct,100.00
median_latency_ms,3863.58
acl_messages,91.00
agent_evaluations,24.00
verified_langsmith_roots,8.00
provider_reported_tokens,17821.00


Metrics apply only to the executed sample, NOT to all 150 cases.


### Optional: verify UI-triggered traces
Run after clicking **Run** in the website. This check does not start Gemini again.

In [29]:
client.flush()
recent = list(client.list_runs(project_name=PROJECT,is_root=True,start_time=RUN_STARTED_AT,limit=100))
ui_traces = [r for r in recent if r.name == 'Studio67-FourAgent-Negotiation' and 'ui' in (r.tags or [])]
print('New UI-triggered traces verified:',len(ui_traces))
for r in ui_traces[:8]:
    print('Case:',(r.inputs or {}).get('case_id'),'Trace:',r.trace_id,'Error:',bool(r.error))
if not ui_traces:
    print('No new UI traces yet. Use the website Run button and execute this check again.')

/tmp/ipykernel_580/3942596911.py:2: LangSmithDeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  recent = list(client.list_runs(project_name=PROJECT,is_root=True,start_time=RUN_STARTED_AT,limit=100))


New UI-triggered traces verified: 4
Case: SCR-017 Trace: 01a1d513-cc82-7113-8b3f-65b5e52ebf67 Error: False
Case: SCR-017 Trace: 01a1d513-b6a1-7c5d-af39-7b15f50a5136 Error: False
Case: SCR-017 Trace: 01a1d513-690d-71cf-9f6a-8bed555bd6c9 Error: False
Case: SCR-001 Trace: 01a1d513-1d79-790d-99ff-978b10a36b7c Error: False


## E — Save evidence (without changing agent files or rerunning Gemini)

In [30]:
RESULTS.mkdir(exist_ok=True)
scenario_df.to_csv(RESULTS/'final_scenarios.csv',index=False)
evaluations_df.to_csv(RESULTS/'final_agent_evaluations.csv',index=False)
acl_df.to_csv(RESULTS/'final_acl_messages.csv',index=False)
trace_df.to_csv(RESULTS/'langsmith_verified_traces.csv',index=False)
span_df.to_csv(RESULTS/'langsmith_nested_spans.csv',index=False)
(RESULTS/'final_metrics.json').write_text(json.dumps(measured,indent=2),encoding='utf-8')
print('Saved measured evidence under:',RESULTS)

Saved measured evidence under: /content/AgenticX/results


In [31]:
import zipfile
EXPORT = Path('/content/AgenticX_Notebook05_Evidence.zip')
required = [
    'integration_api.py','core/observability.py','frontend/index.html',
    'frontend/app.js','agents/creative.py','agents/operations.py'
]
with zipfile.ZipFile(EXPORT,'w',zipfile.ZIP_DEFLATED) as archive:
    for name in required:
        path = ROOT/name
        if path.exists():
            archive.write(path,arcname=name)
    for path in (ROOT/'frontend'/'assets').rglob('*'):
        if path.is_file():
            archive.write(path,arcname=str(path.relative_to(ROOT)))
    for path in RESULTS.glob('final_*'):
        if path.is_file():
            archive.write(path,arcname=str(path.relative_to(ROOT)))
    for path in RESULTS.glob('langsmith_*'):
        if path.is_file():
            archive.write(path,arcname=str(path.relative_to(ROOT)))
print('Evidence ZIP:',EXPORT)
try:
    from google.colab import files
    files.download(str(EXPORT))
except ImportError:
    pass

Evidence ZIP: /content/AgenticX_Notebook05_Evidence.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>